# Notebook 05 : Transgene UMI counts from the alevin-fry matrices

Check for if the `promoter::MYB41` transgene expressed, and does it differ between the
pFACT and pHORST. 

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AF = "/data1/bfernando/DE_project/alvinfry"

# display name -> quant directory
SAMPLES = {
    "Col-0":  "Col-0_quant",
    "pFACT":  "pFACT-MYB41_quant",
    "pHORST": "pHORST-MYB41_quant",
}

TRANSGENE  = "AT4G28110.Fusion"           # promoter::MYB41 construct
ENDOGENOUS = "AT4G28110.Araport11.447"    # native MYB41

def mtx_path(d):  return f"{AF}/{d}/alevin/quants_mat.mtx"
def cols_path(d): return f"{AF}/{d}/alevin/quants_mat_cols.txt"

# check every matrix is present before doing any work
for n, d in SAMPLES.items():
    path = mtx_path(d)
    if os.path.exists(path):
        status = "OK"
    else:
        status = "MISSING"
    print(f"{n:8s} {status:8s} {path}")

## Locate the gene columns

`quants_mat_cols.txt` has one column name per line, so **line number = column index**
(1-based, matching the `.mtx` convention).

USA mode gives **three columns per gene**:

| suffix | meaning |
|---|---|
| *(none)* | **S** — compatible only with the spliced (mature mRNA) form |
| `-U` | **U** — compatible only with the unspliced (intronic / pre-mRNA) form |
| `-A` | **A** — compatible with both, i.e. **ambiguous** |

For single-nucleus data all three are summed — nuclear RNA is largely unspliced, so taking
only S would discard most of the signal.

We look these up **by name** rather than hardcoding indices. The indices happen to be
identical across all three samples (the column order comes from the t2g map, not the reads),
but they would change silently if the reference were ever rebuilt.

In [ ]:
import hashlib

# ---------------------------------------------------------------------------
# Sort the columns of the matrix based on the gene they belong to.
#
# USA mode stores every gene THREE times in the matrix:
#       "GENE"      -> S   (spliced only)
#       "GENE-U"    -> U   (unspliced only)
#       "GENE-A"    -> A   (ambiguous, matches both)
#
# quants_mat_cols.txt gives names of the gene based on the column index from the .mtx file.
# So the LINE NUMBER of a name is its COLUMN NUMBER in the matrix.
# We read the file and note the line number of each of the three names.
# ---------------------------------------------------------------------------
def find_gene_columns(cols_file, gene):

    # the three names we are looking for
    name_S = gene
    name_U = gene + "-U"
    name_A = gene + "-A"

    # not found yet
    col_S = None
    col_U = None
    col_A = None

    line_number = 0
    with open(cols_file) as fh:
        for line in fh:
            line_number = line_number + 1     # 1 based indexing for the .mtc file. 
            name = line.strip()

            if name == name_S:
                col_S = line_number
            elif name == name_U:
                col_U = line_number
            elif name == name_A:
                col_A = line_number

    # If the gene is missing we must stop here. Otherwise we would quietly
    # sum nothing and report 0 UMIs, which looks like "no expression".
    if col_S is None or col_U is None or col_A is None:
        raise ValueError(f"{gene} was not found in {cols_file}")

    return {"S": col_S, "U": col_U, "A": col_A}


cols_ref = cols_path(SAMPLES["Col-0"]) # reading the quants_mat_cols.txt gene names files

TG_COLS = find_gene_columns(cols_ref, TRANSGENE)
EN_COLS = find_gene_columns(cols_ref, ENDOGENOUS)

print(f"{TRANSGENE:30s} {TG_COLS}")
print(f"{ENDOGENOUS:30s} {EN_COLS}")


# ---------------------------------------------------------------------------
# Checking all the 3 counts table to see if the index maps to the same gene.
# Safety check: we look the columns up ONCE (from Col-0) and then use those
# same numbers on all three matrices. That is only valid if all three samples
# have the same column order.
#
# An md5 checksum is a fingerprint of a file. If two files have the same
# checksum they are identical, byte for byte.
# ---------------------------------------------------------------------------
print("\nchecking the column files are identical across samples:")

digests = {}
for name, folder in SAMPLES.items():
    with open(cols_path(folder), "rb") as fh:      # "rb" = read as bytes
        file_bytes = fh.read()
    digests[name] = hashlib.md5(file_bytes).hexdigest()
    print(f"  {name:8s} {digests[name]}")

# compare every sample's checksum against the first one
first_sample = list(SAMPLES.keys())[0]
first_digest = digests[first_sample]

all_same = True
for name in digests:
    if digests[name] != first_digest:
        all_same = False

if all_same:
    print("\n  OK - same column order in every sample, safe to reuse column numbers")
else:
    raise ValueError(f"Column files differ between samples! {digests}")

NameError: name 'cols_path' is not defined

##  Read the matrices

The `.mtx` files are 177-298 MB with 13-22 million non-zero entries. We only need a handful of
columns, so we stream the file line by line rather than loading the whole sparse matrix — one
pass, low memory.

MatrixMarket layout:

```
%%MatrixMarket matrix coordinate real general     <- line 1: format
% written by sprs                                  <- line 2: comment
192259 82968 14268087                              <- line 3: rows cols nonzeros
1 5 2                                              <- line 4+: row col value
```

**`rows` = barcodes, `cols` = genes x 3.** Note Cell Ranger's `.mtx` is the *transpose* of
this (genes x barcodes) 

The three header lines must all be skipped. 

In [5]:
# ---------------------------------------------------------------------------
#
# The .mtx file is huge (up to 22 million lines) but we only need a few
# columns, so we read it ONE LINE AT A TIME instead of loading it all
# into memory.
#
# File layout:
#     line 1:  %%MatrixMarket matrix coordinate real general      <- format
#     line 2:  % written by sprs                                  <- comment
#     line 3:  192259 82968 14268087        <- barcodes, columns, nonzeros
#     line 4+: 1 5 2                        <- barcode, column, UMI count
# ---------------------------------------------------------------------------
def scan_matrix(path, tg_cols, en_cols):

    # Get the UMI counts for the transge
    tg_col_S = tg_cols["S"]
    tg_col_U = tg_cols["U"]
    tg_col_A = tg_cols["A"]

    # Get the UMI counts for the native MYB41
    en_col_S = en_cols["S"]
    en_col_U = en_cols["U"]
    en_col_A = en_cols["A"]

    # running totals for the transgene, one per layer
    tg_S = 0.0
    tg_U = 0.0
    tg_A = 0.0

    # running total for the endogenous gene (all three layers together)
    en_total = 0.0

    # per-barcode dictionaries
    tg_per_bc = {}     # barcode -> transgene UMIs in that barcode
    bc_total = {}      # barcode -> total UMIs in that barcode (all genes)

    with open(path) as fh:

        # --- skip the 3 header lines --------------------------------------
        fh.readline()                       # line 1: format
        fh.readline()                       # line 2: comment
        header = fh.readline()              # line 3: the dimensions

        parts = header.split()
        n_barcodes = int(parts[0])
        n_cols = int(parts[1])
        n_nonzero = int(parts[2])

        # --- read the data, one line at a time ----------------------------
        for line in fh:
            parts = line.split()
            barcode = int(parts[0])         # which row  (barcode)
            column = int(parts[1])          # (gene + layer)
            umis = float(parts[2])          # how many UMIs

            # Count UMIs in each barcode
            if barcode in bc_total:
                bc_total[barcode] = bc_total[barcode] + umis
            else:
                bc_total[barcode] = umis

            # is this one of the transgene columns.
            is_transgene = False
            if column == tg_col_S:
                tg_S = tg_S + umis
                is_transgene = True
            elif column == tg_col_U:
                tg_U = tg_U + umis
                is_transgene = True
            elif column == tg_col_A:
                tg_A = tg_A + umis
                is_transgene = True

            # increment the UMI count for the barcodes carry transgene signal
            if is_transgene:
                if barcode in tg_per_bc:
                    tg_per_bc[barcode] = tg_per_bc[barcode] + umis
                else:
                    tg_per_bc[barcode] = umis

            # is this one of the endogenous MYB41 columns?
            if column == en_col_S or column == en_col_U or column == en_col_A:
                en_total = en_total + umis

    # add up the whole library
    total_umi = 0.0
    for barcode in bc_total:
        total_umi = total_umi + bc_total[barcode]

    results = {
        "n_barcodes": n_barcodes,
        "n_cols": n_cols,
        "n_nonzero": n_nonzero,
        "tg_S": tg_S,
        "tg_U": tg_U,
        "tg_A": tg_A,
        "tg_total": tg_S + tg_U + tg_A,
        "tg_per_bc": tg_per_bc,
        "en_total": en_total,
        "total_umi": total_umi,
        "bc_total": bc_total,
    }
    return results


# run it on all three samples
res = {}
for name, folder in SAMPLES.items():
    print(f"scanning {name} ...", end=" ", flush=True)
    res[name] = scan_matrix(mtx_path(folder), TG_COLS, EN_COLS)
    print("done")

scanning Col-0 ... done
scanning pFACT ... done
scanning pHORST ... done


## The transgene table

In [6]:
# ---------------------------------------------------------------------------
# Build the summary table, one row per sample.
# ---------------------------------------------------------------------------
rows = []
for name in res:
    r = res[name]

    # normalise for sequencing depth: transgene UMIs per million total UMIs
    per_million = r["tg_total"] / r["total_umi"] * 1000000

    row = {
        "sample":             name,
        "barcodes":           r["n_barcodes"],
        "Fusion_UMI":         int(r["tg_total"]),
        "Fusion_barcodes":    len(r["tg_per_bc"]),
        "endogenous_MYB41":   int(r["en_total"]),
        "total_UMI":          int(r["total_umi"]),
        "Fusion_per_million": per_million,
    }
    rows.append(row)

tab = pd.DataFrame(rows)
tab = tab.set_index("sample")

# Col-0 is wild-type and carries no transgene, so its value is the noise floor.
# Express every sample as a multiple of that floor.
floor = tab.loc["Col-0", "Fusion_per_million"]
tab["x_over_WT"] = (tab["Fusion_per_million"] / floor).round(0)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
display(tab)


# ---------------------------------------------------------------------------
# Split the transgene signal by splice layer.
# ---------------------------------------------------------------------------
print("\nUSA breakdown of the transgene:")

usa_rows = []
for name in res:
    r = res[name]
    usa_rows.append({
        "sample":        name,
        "S (spliced)":   int(r["tg_S"]),
        "U (unspliced)": int(r["tg_U"]),
        "A (ambiguous)": int(r["tg_A"]),
        "total":         int(r["tg_total"]),
    })

usa_tab = pd.DataFrame(usa_rows)
usa_tab = usa_tab.set_index("sample")
display(usa_tab)

,barcodes,Fusion_UMI,Fusion_barcodes,endogenous_MYB41,total_UMI,Fusion_per_million,x_over_WT
sample,,,,,,,
Col-0,192259,1,1,6,17592435,0.06,1.00
pFACT,200283,1312,1182,15,29005632,45.23,796.00
pHORST,193477,124,102,7,21357843,5.81,102.00



USA breakdown of the transgene:


,S (spliced),U (unspliced),A (ambiguous),total
sample,,,,
Col-0,1,0,0,1
pFACT,1312,0,0,1312
pHORST,124,0,0,124


## Are the transgene-positive barcodes real cells, or ambient?

The counts above are per **barcode**, and most barcodes are empty droplets carrying only
ambient RNA. In pFACT the ambient soup contains transgene transcript, so some positive
barcodes will be empty droplets rather than expressing nuclei.

The left panel is the headline result. The right panel asks whether the positive barcodes sit
at high total-UMI (plausibly cells) or low total-UMI (probably soup) — which is what cell
calling will have to resolve.

In [ ]:
COLOR = {"Col-0": "#8c8c8c", "pFACT": "#1f4e8c", "pHORST": "#c1666b"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# ---------------------------------------------------------------------------
# LEFT PANEL - depth-normalised transgene signal per sample
# ---------------------------------------------------------------------------
names = []
values = []
colors = []
for name in res:
    names.append(name)
    values.append(tab.loc[name, "Fusion_per_million"])
    colors.append(COLOR[name])

bars = ax1.bar(names, values, color=colors)
ax1.set_ylabel("Fusion UMI per million total UMI")
ax1.set_title("Transgene signal (depth-normalised)")

# write the raw UMI count on top of each bar
for i in range(len(names)):
    bar = bars[i]
    raw_umi = int(tab.loc[names[i], "Fusion_UMI"])
    x = bar.get_x() + bar.get_width() / 2
    ax1.text(x, bar.get_height(), f"{raw_umi:,} UMI",
             ha="center", va="bottom", fontsize=9)

# the wild-type value is the noise floor
ax1.axhline(floor, color="red", linestyle="--", linewidth=1)
ax1.text(len(names) - 0.5, floor, " WT floor",
         color="red", va="bottom", ha="right", fontsize=8)

# ---------------------------------------------------------------------------
# RIGHT PANEL - are the transgene-positive barcodes real cells, or soup?
#
# For every barcode that has ANY transgene signal, look at that barcode's
# TOTAL UMI count. High total = probably a real nucleus.
# Low total  = probably an empty droplet holding ambient RNA.
# ---------------------------------------------------------------------------
for name in res:
    r = res[name]

    if len(r["tg_per_bc"]) == 0:
        continue

    totals = []
    for barcode in r["tg_per_bc"]:
        totals.append(r["bc_total"][barcode])
    totals = np.array(totals)

    label = f"{name}  (n={len(totals):,})"
    ax2.hist(np.log10(totals + 1), bins=40, alpha=0.55,
             label=label, color=COLOR[name])

# 500 UMI was Cell Ranger's lowest called cell in this library
ax2.axvline(np.log10(500), color="k", linestyle=":", linewidth=1)
ax2.text(np.log10(500), ax2.get_ylim()[1] * 0.95, " 500 UMI",
         fontsize=8, va="top")

ax2.set_xlabel("log10(total UMI of that barcode)")
ax2.set_ylabel("transgene-positive barcodes")
ax2.set_title("Are positive barcodes cells or soup?")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# How many transgene-positive barcodes look like plausible cells?
# ---------------------------------------------------------------------------
print("Transgene-positive barcodes above 500 total UMI:")
for name in res:
    r = res[name]

    above_500 = 0
    for barcode in r["tg_per_bc"]:
        if r["bc_total"][barcode] > 500:
            above_500 = above_500 + 1

    total_positive = len(r["tg_per_bc"])
    print(f"  {name:8s} {above_500:5d} / {total_positive:5d}")

## Summary

*(fill in after running — expected values from the command-line extraction)*

| sample | Fusion UMI | Fusion barcodes | endogenous MYB41 | total UMI |
|---|---|---|---|---|
| Col-0 (WT) | 1 | 1 | 6 | 17,592,435 |
| pFACT | 1,312 | 1,182 | 15 | 29,005,632 |
| pHORST | 124 | 102 | 7 | 21,357,843 |

**The transgene is detected in both constructs and effectively absent in wild-type.**

- Col-0 returns **1 UMI** across 192,259 barcodes. It carries no transgene, so this is the
  measured false-positive floor.
- pFACT clears that floor by ~1,300x, pHORST by ~124x. Neither is ambiguous.
- Depth-normalised, **pFACT is ~8x stronger than pHORST**. The two constructs differ only in
  their promoter, so this is a difference in promoter activity.
- Endogenous MYB41 stays low in all three (6 / 15 / 7 UMIs), so transgene signal is not
  bleeding in from the native copy despite the shared MYB41 coding sequence.

**Caveats**

- Barcodes, not cells. The positive-barcode counts are an upper bound; some are empty droplets
  carrying ambient transgene RNA. Detection is unaffected (Col-0 controls for this), but
  **localisation to cell types requires cell calling first**.
- n=1 per genotype. The ~8x difference is large, but with one plant each it cannot be formally
  separated from plant-to-plant variation or insertion-site effects.
- Absolute counts are low (1,312 UMI out of 29M), consistent with a shallow library
  (~80% rRNA) and with MYB41 being stress-inducible.